# Julia notebook to compute the buoyancy field using the 1D vertical diffusion equation, inspired by Peterson & Callies (2025, in review) theory. Then compute the associated geostrophic flow using the frictional thermal wind equation.

twnh Sep '25

This notebook solves the steady vertical diffusion problem with a Green's function.

The problem is to solve:
\begin{align}
\epsilon^2 \kappa \frac{d^2 b}{dz^2} 
    + \gamma \left( B(z) - b \right) & = 0 , 
\end{align}
for $b(z)$, given buoyancy profile $B(z)$, relaxation coefficient $\gamma$, with boundary conditions
\begin{align}
 \frac{db}{dz}  &= 0 \text{~ at~} z = -H , \\
 b &= 0 \text{~ at~} z = 0 ,
\end{align}
where bottom buoyancy gradient is be zero.

This problem is solved repeatedly for different $H(x)$ fields and $B(x,z)$ fields. This constructs a buoyancy field that varies in $(x,z)$ using the 1D diffusion equation.

Then, given $b(x,z)$, solve:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the known wind stress.

In [1]:
using SymPy
using Infiltrator

Define symbols and functions

In [2]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x      = symbols("x",     real=true)                # Horizontal coordinate
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x)
α      = 1//2                                       # Aspect ratio value. Note // which maintains rational type
Hfn    = α * (1 - x^2)                              # Bathymetry function
geometry_params = (H(x), z, ξ, α, x)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀     = symbols("ν₀",    real=true, positive=true) # Viscosity parameters
τˣ, τʸ = symbols("τˣ τʸ", real=true)                # Surface wind stress components

# Define viscosity profile here:
ν = ν₀                                              # Constant viscosity profile

uv_params = (f, ϵ, ν, τˣ, τʸ)

# Buoyancy equation symbolic parameters:
κ₀     = symbols("κ₀",    real=true, positive=true) # Diffusivity parameters
γ      = symbols("γ",     real=true, positive=true) # Relaxation parameter

# Define diffusivity profile here:
κ = κ₀                                              # Constant viscosity profile

b_params = (κ, ϵ, γ) ;

### Set the problem parameters here:

In [3]:
# Define the $B(z)$ source term here:
B      = SymFunction("B")                           # Source term function B(z): buoyancy field relaxation profile.
Bfn  = z                                            # Linear profile with vanishing surface buoyancy

f_val  = 1.0
ϵ_val  = 1.0

γ_val  = 1.0

κ₀_val = 0.01

ν₀_val = κ₀_val

τˣ_val = 0.0
τʸ_val = 0.0
param_values = Dict(f=>f_val, ϵ=>ϵ_val, γ=>γ_val, κ₀=>κ₀_val, ν₀=>ν₀_val, τˣ=>τˣ_val, τʸ=>τʸ_val)

Dict{Sym{PyCall.PyObject}, Float64} with 7 entries:
  γ  => 1.0
  τˣ => 0.0
  κ₀ => 0.01
  f  => 1.0
  ϵ  => 1.0
  ν₀ => 0.01
  τʸ => 0.0

Code to solve the buoyancy equation using a Green's function:

In [4]:
function compute_Gb(b_params, geometry_params, param_values)
    # Setup symbols and parameters:
    κ, ϵ, γ       = b_params
    H, z, ξ, α, x = geometry_params
    b             = SymFunction("b")
    A             = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    im            = SymPy.im  # SymPy's imaginary unit
    
    #0. Define the ODE for b(z):
    ode = Eq(ϵ^2 * κ * diff(diff(b(z),z),z) - diff(κ,z) * diff(b(z),z) - γ * b(z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    
    bm = bm.subs(first(bm_const), A)                              # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = simplify(bm * bp.subs(z,ξ) / (κ.subs(z,ξ) * W.subs(z,ξ)))
    Gp = simplify(bm.subs(z,ξ) * bp / (κ.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/κ.subs(z,ξ) == 0

    # Define piecewise Green's function:
    G = simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

compute_Gb (generic function with 1 method)

Code to solve the frictional geostrophic equation using a Green's function:

In [5]:
function compute_Guv(uv_params, geometry_params, param_values)
    # Setup symbols and parameters:
    f, ϵ, ν, τˣ, τʸ = uv_params
    H, z, ξ, α, x   = geometry_params
    uv              = SymFunction("uv")
    A               = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    im              = SymPy.im                                         # SymPy's imaginary unit

    #0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x))=>0)).rhs
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert simplify(Gₘ.subs(z,-H(x))) == 0                                     # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs         # This is where surface forcing would go.
    @assert simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = simplify(Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z))

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = simplify(Gₘ * Gₚ.subs(z,ξ) / (ν.subs(z,ξ) * W.subs(z,ξ)))
    Gp = simplify(Gₘ.subs(z,ξ) * Gₚ / (ν.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/ν.subs(z,ξ) == 0

    # #5. Define piecewise Green's function:
    G = simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))
    
    # Check boundary conditions:
    @assert simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert simplify(G.subs(z,-H(x)).subs(ξ,-H//2)) == 0

    return G
end

compute_Guv (generic function with 1 method)

# Compute the G's functions:

In [6]:
Gb_sym  = compute_Gb(b_params,geometry_params,param_values) 
display("Gb(x,ξ):")
display(Gb_sym)
Guv_sym = compute_Guv(uv_params,geometry_params,param_values) 
display("Guv(x,ξ):")
display(Guv_sym)

"Gb(x,ξ):"

/  /          ___   \ /     ___                ___ \    ___                    
|  |     -2*\/ γ *ξ | | 2*\/ γ *H(x)    -2*z*\/ γ  |  \/ γ *(z + ξ)            
|  |     -----------| | ------------    -----------|  -------------            
|  |        ____    | |     ____           ____    |      ____                 
|  |      \/ κ₀ *ϵ  | |   \/ κ₀ *ϵ       \/ κ₀ *ϵ  |    \/ κ₀ *ϵ               
|ϵ*\1 - e           /*\e             + e           /*e                         
|------------------------------------------------------------------  for z <= ξ
|                               /     ___         \                            
|                               | 2*\/ γ *H(x)    |                            
|                               | ------------    |                            
|                               |     ____        |                            
|                    ___   ____ |   \/ κ₀ *ϵ      |                            
|                2*\/ γ *\/ κ₀ *\e      

"Guv(x,ξ):"

/  /         ___   ____           \ /     ___     ____    \    ___     5/2     >
|  |     2*\/ f *\/ -I *(z + H(x))| | 2*\/ f *ξ*\/ -I     |  \/ f *(-I)   *(z  >
|  |     -------------------------| | ----------------    |  ----------------- >
|  |               ____           | |       ____          |          ____      >
|  |             \/ ν₀ *ϵ         | |     \/ ν₀ *ϵ        |        \/ ν₀ *ϵ    >
|ϵ*\1 - e                         /*\e                 + 1/*e                  >
|----------------------------------------------------------------------------- >
|                                      /     ___   ____         \              >
|                                      | 2*\/ f *\/ -I *H(x)    |              >
|                                      | -------------------    |              >
|                                      |        ____            |              >
|                    ___   ____   ____ |      \/ ν₀ *ϵ          |              >
|                2*\/ f *\/ 

### For LaTeX write up:

#### Solve for the buoyancy profile $b(z)$ and its derivative $\partial_x b$ using the buoyancy Green's function:

In [7]:
# Be careful with this function!!!
# It's more powerful than the .subs() method, but is more risky.
function tom_subs(expr,param_values)
    new_expr_str = string(expr)
    for (k, v) in param_values
        new_expr_str = replace(new_expr_str, string(k) => string(v))
    end
    return sympify(new_expr_str)
end

function get_var_in_expr(var,expr)
    filter(s -> string(s) == var, collect(expr.free_symbols))[1]
end

# Compute full symbolic expressions with no substitutions:
b_sym = simplify(integrate(-Gb_sym * B(ξ), (ξ,-H(x),0) ))            # Notice the horizontal dependence in H.
display("Simplified b(z,ξ)")
display(b_sym)
display("Simplified d/dx b(z,ξ)")
db_symdx = diff(b_sym(x),x)
display(db_symdx)

"Simplified b(z,ξ)"

  /                  /               -H(x)                     -H(x)           >
  |                  |                 /                         /             >
  |                  |                |                         |              >
  |  /         ___ \ |     ___        |           ___           |            _ >
  |  |     z*\/ γ  | | 2*\/ γ *H(x)   |         \/ γ *ξ         |         -\/  >
  |  |     --------| | ------------   |         --------        |         ---- >
  |  |       ____  | |     ____       |           ____          |           __ >
  |  |     \/ κ₀ *ϵ| |   \/ κ₀ *ϵ     |         \/ κ₀ *ϵ        |         \/ κ >
ϵ*|- \1 - e        /*|e            *  |   B(ξ)*e         dξ +   |   B(ξ)*e     >
  |                  |                |                         |              >
  |                  |               /                         /               >
  \                  \                                                         >
----------------------------

"Simplified d/dx b(z,ξ)"

                                                                               >
                                                                               >
                                                                               >
  /                  /               -H(x)                     -H(x)           >
  |                  |                 /                         /             >
  |                  |                |                         |              >
  |  /         ___ \ |     ___        |           ___           |            _ >
  |  |     z*\/ γ  | | 2*\/ γ *H(x)   |         \/ γ *ξ         |         -\/  >
  |  |     --------| | ------------   |         --------        |         ---- >
  |  |       ____  | |     ____       |           ____          |           __ >
  |  |     \/ κ₀ *ϵ| |   \/ κ₀ *ϵ     |         \/ κ₀ *ϵ        |         \/ κ >
  |- \1 - e        /*|e            *  |   B(ξ)*e         dξ +   |   B(ξ)*e     >
  |                  |      

$G_b$ with compound constant

In [8]:
kappa_var   = get_var_in_expr("κ₀",Gb_sym)
epsilon_var = get_var_in_expr("ϵ" ,Gb_sym)
gamma_var   = get_var_in_expr("γ" ,Gb_sym)
ϕ = gamma_var/(epsilon_var^2 * kappa_var)

tmp  = tom_subs(Gb_sym,Dict("γ"=>"ϕ^2 * ϵ^2 * κ₀ "))               # THIS IS FRAGILE!!!
tmp2 = tom_subs(tmp,Dict("sqrt(κ₀*ϕ^2*ϵ^2)/(sqrt(κ₀)*ϵ)"=>"ϕ"))
tmp3 = tom_subs(tmp2,Dict("sqrt(κ₀*ϕ^2*ϵ^2)"=>"ϕ*sqrt(κ₀)*ϵ"))
display("Simplified Gb(z,ξ)")
display(tmp3)

tmp4 = tom_subs(tmp3,Dict("ξ"=>"-H(x)"))
display("Simplified Gb(z,-H(x))")
display(simplify(tmp4.args[1].args[1]))

display("Simplified d/dx Gb(z,ξ)")
x_var = get_var_in_expr("x" ,tmp3)
tmp5  = simplify(diff(tmp3,x_var))
display(tmp5)

"Simplified Gb(z,ξ)"

//     -2*ξ*ϕ\ / 2*ϕ*H(x)    -2*z*ϕ\  ϕ*(z + ξ)            
|\1 - e      /*\e         + e      /*e                     
|----------------------------------------------  for z <= ξ
|                   / 2*ϕ*H(x)    \                        
|            2*κ₀*ϕ*\e         + 1/                        
<                                                          
|/     -2*z*ϕ\ / 2*ϕ*H(x)    -2*ξ*ϕ\  ϕ*(z + ξ)            
|\1 - e      /*\e         + e      /*e                     
|----------------------------------------------  for z >= ξ
|                   / 2*ϕ*H(x)    \                        
\            2*κ₀*ϕ*\e         + 1/                        

"Simplified Gb(z,-H(x))"

-cosh(ϕ*(z + H(x)))*tanh(ϕ*H(x)) 
---------------------------------
              κ₀*ϕ               

"Simplified d/dx Gb(z,ξ)"

//   2*z*ϕ    2*ξ*ϕ    2*ϕ*(z + ξ)    \  ϕ*(-z - ξ + 2*H(x)) d                 >
|\- e      - e      + e            + 1/*e                   *--(H(x))          >
|                                                            dx                >
<--------------------------------------------------------------------  for Or( >
|                     / 4*ϕ*H(x)      2*ϕ*H(x)    \                            >
|                  κ₀*\e         + 2*e         + 1/                            >
\                                                                              >

>                
>                
>                
> z >= ξ, z <= ξ)
>                
>                
>                

$G_{uv}$ with compound constant

In [9]:
nu_var      = get_var_in_expr("ν₀",Guv_sym)
epsilon_var = get_var_in_expr("ϵ" ,Guv_sym)
f_var       = get_var_in_expr("f" ,Guv_sym)
ϕ           = f_var/(epsilon_var^2 * nu_var)

tmp  = tom_subs(Guv_sym,Dict("f"=>"ϕ^2 * ϵ^2 * ν₀"))               # THIS IS FRAGILE!!!
tmp2 = tom_subs(tmp,Dict("sqrt(ν₀*ϕ^2*ϵ^2)/(sqrt(ν₀)*ϵ)"=>"ϕ"))
tmp3 = tom_subs(tmp2,Dict("sqrt(ν₀*ϕ^2*ϵ^2)"=>"ϕ*sqrt(ν₀)*ϵ"))
display("Simplified Guv(z,ξ)")
display(tmp3)

z_var = get_var_in_expr("z" ,tmp3)
tmp4  = tom_subs(tmp3,Dict("ξ"=>"0"))
display("Simplified Guv(z,0)")
display(simplify(tmp4.args[1].args[1]))

tmp5  = diff(tmp3, z_var)
tmp6 = tom_subs(tmp5,Dict(z_var=>"-H(x)"))
display("Simplified d/dz Guv(z,ξ) @ z = -H(x)")
display(simplify(tmp6.args[1].args[1]))

tmp7 = integrate(tmp3, (z_var, -H(x), 0))
# expr = simplify(tmp7.args[1].args[1])
expr = tmp7.args[1].args[1]
expr = tom_subs(expr,Dict("Max(ξ, -H(x))"=>"ξ"))
expr = tom_subs(expr,Dict("Min(0, ξ)"=>"ξ"))
expr = simplify(factor(expr))
display("Simplified integral Guv(z,ξ) wrt z = -H(x) to 0")
display(expr)

xi_var = get_var_in_expr("ξ" ,expr)
display("Simplified integral Guv(z,ξ) wrt z = -H(x) to 0 @ ξ = 0")
tmp8 = tom_subs(expr,Dict(xi_var=>"0"))
display(simplify(factor(tmp8)))

"Simplified Guv(z,ξ)"

//           ____           \ /         ____    \        5/2                    
||     2*ϕ*\/ -I *(z + H(x))| | 2*ξ*ϕ*\/ -I     |  ϕ*(-I)   *(z + ξ)            
|\1 - e                     /*\e             + 1/*e                             
|-------------------------------------------------------------------  for z <= ξ
|                             /       ____         \                            
|                        ____ | 2*ϕ*\/ -I *H(x)    |                            
|               2*ν₀*ϕ*\/ -I *\e                + 1/                            
<                                                                               
|/           ____           \ /         ____    \        5/2                    
||     2*ϕ*\/ -I *(ξ + H(x))| | 2*z*ϕ*\/ -I     |  ϕ*(-I)   *(z + ξ)            
|\1 - e                     /*\e             + 1/*e                             
|-------------------------------------------------------------------  for z >= ξ
|                           

"Simplified Guv(z,0)"

/           ____           \          5/2
|     2*ϕ*\/ -I *(z + H(x))|  z*ϕ*(-I)   
\1 - e                     /*e           
-----------------------------------------
               /       ____         \    
          ____ | 2*ϕ*\/ -I *H(x)    |    
   ν₀*ϕ*\/ -I *\e                + 1/    

"Simplified d/dz Guv(z,ξ) @ z = -H(x)"

 /         ____    \        5/2            
 | 2*ξ*ϕ*\/ -I     |  ϕ*(-I)   *(ξ - H(x)) 
-\e             + 1/*e                     
-------------------------------------------
            /       ____         \         
            | 2*ϕ*\/ -I *H(x)    |         
         ν₀*\e                + 1/         

"Simplified integral Guv(z,ξ) wrt z = -H(x) to 0"

  /       ____        ____     \ /     ____               \          5/2
  | ξ*ϕ*\/ -I     ϕ*\/ -I *H(x)| | ϕ*\/ -I *(ξ + H(x))    |  ξ*ϕ*(-I)   
I*\e           - e             /*\e                    - 1/*e           
------------------------------------------------------------------------
                            /       ____         \                      
                          2 | 2*ϕ*\/ -I *H(x)    |                      
                      ν₀*ϕ *\e                + 1/                      

"Simplified integral Guv(z,ξ) wrt z = -H(x) to 0 @ ξ = 0"

                        2   
    /     ____         \    
    | ϕ*\/ -I *H(x)    |    
 -I*\e              - 1/    
----------------------------
      /       ____         \
    2 | 2*ϕ*\/ -I *H(x)    |
ν₀*ϕ *\e                + 1/